# Meta-Optical Encoder for Image Segmentation

Official notebook for the paper: **Meta-Optical Encoder for Image Segmentation**  


# 1. Install dependencies

In [ ]:
from os.path import splitext
from os import listdir
from glob import glob
import logging

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch import optim
from tqdm import tqdm


from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, random_split

import copy
from typing import Optional, Tuple

# 2. Define the CarvanaDataset

In [ ]:
class BasicDataset(Dataset):
    """
    Basic PyTorch dataset for image segmentation.

    This dataset assumes a paired folder structure:
        - imgs_dir: directory containing input images
        - masks_dir: directory containing corresponding masks

    Each sample is identified by a unique file stem (filename without extension).
    For a given ID, the dataset searches:
        image:  imgs_dir/{id}.*
        mask:   masks_dir/{id}_mask.*   (default suffix="_mask")

    Notes:
        - Images and masks are resized according to `scale` (0 < scale <= 1).
        - Returned tensors are float32 and normalized to [0, 1] if needed.
        - Output format:
            {
              "image": FloatTensor [C, H, W],
              "mask":  FloatTensor [C, H, W] or [1, H, W] depending on mask file
            }
    """

    def __init__(self, imgs_dir, masks_dir, scale=1, mask_suffix=''):
        """
        Args:
            imgs_dir (str): Path to the directory containing images.
            masks_dir (str): Path to the directory containing masks.
            scale (float): Resize ratio applied to both image and mask.
            mask_suffix (str): Suffix appended to ID for locating mask files.
                               Example: id="abc" -> mask file "abc{mask_suffix}.*"
        """
        self.imgs_dir = imgs_dir
        self.masks_dir = masks_dir
        self.scale = scale
        self.mask_suffix = mask_suffix

        assert 0 < scale <= 1, "Scale must be between 0 and 1"

        # Collect all IDs from image directory (ignore hidden files)
        self.ids = [
            splitext(file)[0]
            for file in listdir(imgs_dir)
            if not file.startswith('.')
        ]
        logging.info(f"Creating dataset with {len(self.ids)} examples")

    def __len__(self):
        """Return the number of samples in the dataset."""
        return len(self.ids)

    @classmethod
    def preprocess(cls, pil_img, scale, mask_flag):
        """
        Preprocess a PIL image (either input image or segmentation mask).

        Steps:
            1) Resize according to the given `scale`
            2) Convert to numpy array
            3) Ensure 3D shape (H, W, C)
            4) Convert from HWC -> CHW
            5) Normalize into [0, 1] if pixel range appears to be [0, 255]

        Args:
            pil_img (PIL.Image): Input PIL image.
            scale (float): Resize ratio.
            mask_flag (bool): Whether this is a mask (currently same resize logic).
                              You can customize mask handling here if needed.

        Returns:
            np.ndarray: Float image array in CHW format.
        """
        w, h = pil_img.size
        newW, newH = int(scale * w), int(scale * h)

        assert newW > 0 and newH > 0, "Scale is too small"

        # Resize (same behavior for mask and image in current implementation)
        pil_img = pil_img.resize((newW, newH))

        img_nd = np.array(pil_img)

        # If grayscale image/mask: (H, W) -> (H, W, 1)
        if len(img_nd.shape) == 2:
            img_nd = np.expand_dims(img_nd, axis=2)

        # HWC -> CHW (channel-first for PyTorch)
        img_trans = img_nd.transpose((2, 0, 1))

        # Normalize only when max is > 1 (assume 8-bit image)
        if img_trans.max() > 1:
            img_trans = img_trans / 255.0

        return img_trans

    def __getitem__(self, i):
        """
        Load the i-th sample.

        Args:
            i (int): index

        Returns:
            dict: {
                "image": torch.FloatTensor [C, H, W],
                "mask":  torch.FloatTensor [C, H, W]
            }
        """
        idx = self.ids[i]

        # Find mask and image files via glob
        mask_file = glob(self.masks_dir + idx + '_mask.*')
        img_file = glob(self.imgs_dir + idx + '.*')

        # Ensure exactly one match is found
        assert len(mask_file) == 1, (
            f"Either no mask or multiple masks found for the ID {idx}: {mask_file}"
        )
        assert len(img_file) == 1, (
            f"Either no image or multiple images found for the ID {idx}: {img_file}"
        )

        # Load files
        mask = Image.open(mask_file[0])
        img = Image.open(img_file[0])

        # Ensure image and mask match spatial dimensions
        assert img.size == mask.size, (
            f"Image and mask {idx} should be the same size, but are {img.size} and {mask.size}"
        )

        # Preprocess
        img = self.preprocess(img, self.scale, mask_flag=False)
        mask = self.preprocess(mask, self.scale, mask_flag=True)

        return {
            "image": torch.from_numpy(img).float(),
            "mask": torch.from_numpy(mask).float()
        }


class CarvanaDataset(BasicDataset):
    """
    Carvana dataset wrapper.

    This dataset uses the standard naming rule:
        - image: {id}.*
        - mask:  {id}_mask.*

    It inherits from BasicDataset and sets mask suffix accordingly.
    """
    def __init__(self, imgs_dir, masks_dir, scale=1):
        super().__init__(imgs_dir, masks_dir, scale, mask_suffix="_mask")


In [ ]:
import torch
from torch.autograd import Function


class DiceCoeff(Function):
    """
    This implementation:
        - Computes Dice score for one prediction/target pair.
        - Supports backward gradient w.r.t. the input (prediction).
        - Assumes `input` and `target` have the same shape.

    Args:
        input (torch.Tensor): predicted segmentation map. Usually float tensor
                              with values in [0, 1] (e.g. sigmoid output).
        target (torch.Tensor): ground-truth segmentation map (binary mask).
    """

    def forward(self, input, target):
        """
        Forward pass: compute Dice coefficient.

        Notes:
            - `eps` avoids division by zero.
            - Flattens tensors to 1D before dot product.

        Returns:
            torch.Tensor: scalar Dice coefficient for the sample.
        """
        self.save_for_backward(input, target)
        eps = 1e-4

        # Intersection and union (flatten to 1D)
        self.inter = torch.dot(input.view(-1), target.view(-1))
        self.union = torch.sum(input) + torch.sum(target) + eps

        dice = (2 * self.inter.float() + eps) / self.union.float()
        return dice

    def backward(self, grad_output):
        """
        Backward pass: compute gradient for Dice coefficient.

        Returns:
            grad_input (torch.Tensor | None): gradient w.r.t input
            grad_target: always None (ground truth has no gradient)
        """
        input, target = self.saved_variables
        grad_input = grad_target = None

        if self.needs_input_grad[0]:
            # d/dx Dice(x,y) for input x
            grad_input = grad_output * 2 * (target * self.union - self.inter) / (self.union * self.union)

        # ground-truth target is constant
        if self.needs_input_grad[1]:
            grad_target = None

        return grad_input, grad_target


def dice_coeff(input, target):

    # Initialize accumulator on the same device
    s = torch.zeros(1, device=input.device, dtype=torch.float32)

    for i, (inp, tgt) in enumerate(zip(input, target)):
        s = s + DiceCoeff().forward(inp, tgt)

    return s / (i + 1)


# 3. Teacher and Student Model

In [ ]:
"""
Parts of the U-Net model
-----------------------

This file implements the building blocks used in a U-Net-style segmentation network.

The components include:
    - DoubleConv: basic convolution block
    - Down: encoder down-sampling block
    - Up: decoder up-sampling block with skip connection
    - OutConv: final 1x1 output projection layer


Attribution
-----------
This implementation is adapted/modified from the following public repository:

https://github.com/milesial/Pytorch-UNet
"""


class DoubleConv(nn.Module):
    """
    Double convolution block commonly used in U-Net.

    Structure:
        Conv2d -> BatchNorm2d -> ReLU
        (applied twice in standard U-Net)

    Note:
        In this implementation the sequential currently contains only ONE Conv-BN-ReLU.
        If you want the canonical U-Net, you should duplicate the Conv-BN-ReLU block.

    Args:
        in_channels (int): number of input channels.
        out_channels (int): number of output channels.
        mid_channels (int, optional): intermediate channel dimension. If None, defaults to out_channels.

    Input:
        x (torch.Tensor): shape [B, in_channels, H, W]

    Output:
        torch.Tensor: shape [B, out_channels (or mid_channels), H, W]
    """

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        """
        Forward pass.

        Args:
            x (torch.Tensor): input feature map.

        Returns:
            torch.Tensor: transformed feature map.
        """
        return self.double_conv(x)


class Down(nn.Module):
    """
    Down-sampling block used in the encoder path of U-Net.

    Structure:
        MaxPool2d(2) -> DoubleConv

    This reduces spatial resolution by a factor of 2.

    Args:
        in_channels (int): number of input channels.
        out_channels (int): number of output channels.

    Input:
        x (torch.Tensor): shape [B, in_channels, H, W]

    Output:
        torch.Tensor: shape [B, out_channels, H/2, W/2]
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        """
        Forward pass.

        Args:
            x (torch.Tensor): input feature map.

        Returns:
            torch.Tensor: down-sampled feature map.
        """
        return self.maxpool_conv(x)


class Up(nn.Module):
    """
    Up-sampling block used in the decoder path of U-Net.

    The decoder upsamples a low-resolution feature map and merges it with a
    corresponding skip-connection feature map from the encoder.

    Two upsampling modes are supported:
        - bilinear upsampling + convolution projection
        - transposed convolution

    Args:
        in_channels (int): number of input channels after concatenation.
        out_channels (int): number of output channels.
        bilinear (bool): whether to use bilinear upsampling. If False, use ConvTranspose2d.

    Inputs:
        x1 (torch.Tensor): decoder feature map to be upsampled, shape [B, C1, H1, W1]
        x2 (torch.Tensor): skip feature map from encoder, shape [B, C2, H2, W2]

    Output:
        torch.Tensor: fused feature map, shape [B, out_channels, H2, W2]

    Notes:
        Due to possible mismatch in spatial size after upsampling, padding is applied
        before concatenation to align shapes.
    """

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            # Bilinear upsampling (no learnable parameters)
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

            # Reduce channels using convolution after concatenation
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            # Learnable upsampling via transposed convolution
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        """
        Forward pass with skip connection.

        Args:
            x1 (torch.Tensor): decoder feature to upsample.
            x2 (torch.Tensor): encoder skip feature.

        Returns:
            torch.Tensor: output feature map after upsampling + concatenation + conv.
        """
        x1 = self.up(x1)

        # Compute spatial size difference (H, W)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        # Pad x1 to match x2 size
        x1 = F.pad(
            x1,
            [
                diffX // 2, diffX - diffX // 2,
                diffY // 2, diffY - diffY // 2
            ]
        )

        # Concatenate along channel dimension
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    """
    Output convolution block.

    This is typically the last layer of U-Net, projecting feature maps to
    segmentation logits.

    Structure:
        Conv2d(kernel_size=1)

    Args:
        in_channels (int): number of input channels.
        out_channels (int): number of output channels (usually #classes).

    Input:
        x (torch.Tensor): shape [B, in_channels, H, W]

    Output:
        torch.Tensor: shape [B, out_channels, H, W] (logits)
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        """
        Forward pass.

        Args:
            x (torch.Tensor): input feature map.

        Returns:
            torch.Tensor: output logits map.
        """
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net model for image segmentation (teacher model).

    This implementation follows the classic U-Net encoder-decoder architecture with
    skip connections, adapted from:
        https://github.com/milesial/Pytorch-UNet

    Architecture overview:
        Input -> [Encoder: Down-sampling path] -> Bottleneck ->
              -> [Decoder: Up-sampling path + skip connections] -> Output logits

    Args:
        n_channels (int): Number of input image channels (e.g., 3 for RGB, 1 for grayscale).
        n_classes (int): Number of output segmentation classes.
                         - If n_classes=1: typically binary segmentation with sigmoid.
                         - If n_classes>1: multi-class segmentation with softmax.
        bilinear (bool): If True, use bilinear upsampling in the decoder.
                         If False, use transposed convolution (ConvTranspose2d).

    Input:
        x (torch.Tensor): shape [B, n_channels, H, W]

    Output:
        logits (torch.Tensor): shape [B, n_classes, H, W]
            Raw logits (not activated). Apply:
                - sigmoid for binary segmentation
                - softmax for multi-class segmentation

    Notes:
        - This model is commonly used as a high-capacity "teacher" network for
          knowledge distillation experiments.
        - Spatial padding is handled within Up blocks to ensure tensor alignment.
    """

    def __init__(self, n_channels=3, n_classes=1, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Encoder (contracting path)
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)

        # Bottleneck: if using bilinear upsampling, reduce feature channels by factor=2
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)

        # Decoder (expanding path)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)

        # Output projection
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        """
        Forward pass of U-Net.

        Encoder extracts multi-scale features; decoder upsamples and fuses them
        with corresponding encoder activations via skip connections.

        Args:
            x (torch.Tensor): input tensor [B, n_channels, H, W]

        Returns:
            torch.Tensor: segmentation logits [B, n_classes, H, W]
        """
        # Encoder path
        x1 = self.inc(x)       # [B, 64, H, W]
        x2 = self.down1(x1)    # [B, 128, H/2, W/2]
        x3 = self.down2(x2)    # [B, 256, H/4, W/4]
        x4 = self.down3(x3)    # [B, 512, H/8, W/8]
        x5 = self.down4(x4)    # [B, 1024//factor, H/16, W/16]

        # Decoder path + skip connections
        x = self.up1(x5, x4)   # fuse with encoder x4
        x = self.up2(x, x3)    # fuse with encoder x3
        x = self.up3(x, x2)    # fuse with encoder x2
        x = self.up4(x, x1)    # fuse with encoder x1

        logits = self.outc(x)  # [B, n_classes, H, W]
        return logits

    def use_checkpointing(self):
        """
        Enable gradient checkpointing to reduce GPU memory usage.

        Checkpointing trades compute for memory:
            - Saves memory by not storing intermediate activations.
            - Recomputes activations during backward pass.

        This is useful for:
            - large input resolutions
            - large batch sizes
            - limited GPU memory

        Warning:
            - This will slow down training (extra forward computations).
            - Requires the blocks to be "pure functions" of inputs.
        """
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.down3 = torch.utils.checkpoint(self.down3)
        self.down4 = torch.utils.checkpoint(self.down4)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.up3 = torch.utils.checkpoint(self.up3)
        self.up4 = torch.utils.checkpoint(self.up4)
        self.outc = torch.utils.checkpoint(self.outc)



In [ ]:
import torch
import torch.nn as nn


class P_UNet(nn.Module):
    """
    Lightweight / shallow U-Net variant for segmentation (student model).

    This model is a simplified U-Net with:
        - One initial feature extraction block (DoubleConv)
        - Two down-sampling blocks (Down)
        - Two up-sampling blocks (Up) with skip connections
        - One final 1x1 output projection (OutConv)

    The goal of this architecture is typically to serve as a compact student network
    (e.g., for knowledge distillation from a larger teacher U-Net).

    Args:
        n_channels (int): Number of input image channels (default=3 for RGB).
        n_classes (int): Number of segmentation output channels/classes.
                         - 1 for binary segmentation (use sigmoid at inference)
                         - >1 for multi-class segmentation (use softmax at inference)
        bilinear (bool): Upsampling mode. If True, use bilinear interpolation;
                         otherwise use transposed convolution.

    Input:
        x (torch.Tensor): shape [B, n_channels, H, W]

    Output:
        logits (torch.Tensor): shape [B, n_classes, H, W]
            Raw logits (no activation). Apply sigmoid/softmax outside depending on task.
    """

    def __init__(self, n_channels=3, n_classes=1, bilinear=False):
        super(P_UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Initial convolution block (feature extraction)
        self.inc = DoubleConv(n_channels, 8)

        # Down-sampling blocks (encoder)
        # NOTE: The current implementation uses `n_channels` as the input channels here.
        # If you want a canonical U-Net, these should typically be Down(8, 16) and Down(16, 32//factor).
        self.down1 = Down(n_channels, 16)
        self.down2 = Down(n_channels, 32 // (2 if bilinear else 1))

        # Up-sampling blocks (decoder)
        self.up1 = Up(32, 16 // (2 if bilinear else 1), bilinear)
        self.up2 = Up(16, 8, bilinear)

        # Output projection (logits)
        self.outc = OutConv(8, n_classes)

    def forward(self, x):
        """
        Forward pass.

        The network follows an encoder-decoder structure:
            x -> inc -> down1/down2 -> up1/up2 -> outc

        Args:
            x (torch.Tensor): input tensor [B, n_channels, H, W]

        Returns:
            torch.Tensor: segmentation logits [B, n_classes, H, W]
        """
        # Contracting path (downsampling / encoder)
        x1 = self.inc(x)        # low-level features
        x2 = self.down1(x)      # first down-sampled features

        # Bottleneck / deeper features
        x3 = self.down2(x)      # second down-sampled features

        # Expansive path (upsampling / decoder) with skip connections
        x = self.up1(x3, x2)    # skip connection with x2
        x = self.up2(x, x1)     # skip connection with x1

        # Output logits
        logits = self.outc(x)
        return logits

    def use_checkpointing(self):
        """
        Enable gradient checkpointing to save GPU memory.

        This trades compute for memory by recomputing intermediate activations
        during backward pass. Useful for large images or limited GPU memory.
        """
        self.inc = torch.utils.checkpoint(self.inc)
        self.down1 = torch.utils.checkpoint(self.down1)
        self.down2 = torch.utils.checkpoint(self.down2)
        self.up1 = torch.utils.checkpoint(self.up1)
        self.up2 = torch.utils.checkpoint(self.up2)
        self.outc = torch.utils.checkpoint(self.outc)

In [ ]:
def eval_net(net, loader, device):
    """
    Evaluate a segmentation network on a set.

    This function runs inference on the dataloader without using DenseCRF
    post-processing, and computes an evaluation score depending on task type:

    - Multi-class segmentation (net.n_classes > 1):
        Returns the average CrossEntropyLoss over all batches.

    - Binary segmentation (net.n_classes == 1):
        Applies sigmoid + threshold (0.5) to obtain binary predictions, then returns
        the average Dice coefficient over all batches.

    Args:
        net (torch.nn.Module): segmentation model. Must have attribute `n_classes`.
        loader (torch.utils.data.DataLoader): dataloader. Each batch is a dict:
            {
                "image": Tensor [B, C, H, W],
                "mask":  Tensor [B, 1, H, W] or [B, H, W] depending on dataset
            }
        device (torch.device or str): evaluation device ("cuda" or "cpu").

    Returns:
        float: score averaged over batches.
            - If multi-class: mean cross-entropy loss (lower is better).
            - If binary: mean Dice coefficient (higher is better).

    Notes:
        - The model is temporarily set to eval() mode and restored to train() at the end.
        - Uses torch.no_grad() to save memory and computation during evaluation.
        - Dice is computed on thresholded predictions (hard masks), not soft probabilities.
    """
    net.eval()

    # Ground-truth mask dtype depends on whether it is binary or multi-class segmentation
    mask_type = torch.float32 if net.n_classes == 1 else torch.long

    n_val = len(loader)  # number of batches
    tot = 0

    with tqdm(
        total=n_val,
        desc="Validation round",
        unit="batch",
        leave=True,
        position=0
    ) as pbar:
        for batch in loader:
            imgs, true_masks = batch["image"], batch["mask"]

            # Move to device
            imgs = imgs.to(device=device, dtype=torch.float32)
            true_masks = true_masks.to(device=device, dtype=mask_type)

            # Forward pass (no gradients)
            with torch.no_grad():
                mask_pred = net(imgs)

            # Multi-class case: compute CrossEntropyLoss on logits
            if net.n_classes > 1:
                tot += F.cross_entropy(mask_pred, true_masks).item()

            # Binary case: sigmoid -> threshold -> Dice
            else:
                pred = torch.sigmoid(mask_pred)
                pred = (pred > 0.5).float()
                tot += dice_coeff(pred, true_masks).item()

            pbar.update()

    # Restore training mode
    net.train()

    return tot / n_val


# 4. NTKD

In [ ]:
"""
This function is adapted from the NTK-SAP repository:
https://github.com/YiteWang/NTK-SAP

Specifically, the implementation follows the NTK-SAP pruning method (NTKSAP pruner),
which estimates the NTK trace proxy (||J||_F^2) using a finite-difference perturbation:
it perturbs model parameters by epsilon * N(0, I), computes the output difference
between the original and perturbed model, and uses ||f(x;θ)-f(x;θ+Δθ)||^2 as a proxy
for NTK.

"""

def compute_ntk_proxy_fd_grad(
    model: nn.Module,
    x: torch.Tensor,
    *,
    epsilon: float = 1e-3,
    R: int = 1,
    normalize: bool = True,
) -> torch.Tensor:
    """
    Differentiable NTK trace proxy (finite-difference).
    Adapted from NTK-SAP idea: compare f(x;θ) vs f(x;θ+εz) and use ||diff||^2.

    Returns:
      tensor WITH gradients w.r.t. model parameters.
    """
    try:
        from torch.func import functional_call
    except Exception as e:
        raise RuntimeError("compute_ntk_proxy_fd_grad requires PyTorch 2.x (torch.func.functional_call).") from e

    # IMPORTANT: keep gradient for model params (do NOT use torch.no_grad here)
    model.eval()

    params = dict(model.named_parameters())
    buffers = dict(model.named_buffers())

    out_orig = model(x)
    out_orig = out_orig.view(out_orig.shape[0], -1)  # (B, D)

    acc = 0.0
    for _ in range(R):
        # noise does NOT need grad
        noise = {k: torch.randn_like(v) for k, v in params.items()}
        perturbed_params = {k: (v + epsilon * noise[k]) for k, v in params.items()}

        out_mod = functional_call(model, (perturbed_params, buffers), (x,))
        out_mod = out_mod.view(out_mod.shape[0], -1)

        diff = out_orig - out_mod
        jac_approx = diff.pow(2).squeeze(1)  # (B, H, W)


        # if normalize:
        #     jac_approx = jac_approx / (epsilon ** 2)

        acc = acc + jac_approx

    return acc / R

def compute_ntkd_loss(
    teacher: nn.Module,
    student: nn.Module,
    x_ntk: torch.Tensor,
    *,
    epsilon: float = 1e-3,
    R: int = 1,
    normalize: bool = True
) -> torch.Tensor:
    """
    NTKD loss: match student NTK proxy to teacher NTK proxy.
    """
    with torch.no_grad():
        ntk_t = compute_ntk_proxy_fd_grad(teacher, x_ntk, epsilon=epsilon, R=R, normalize=normalize).detach()

    ntk_s = compute_ntk_proxy_fd_grad(student, x_ntk, epsilon=epsilon, R=R, normalize=normalize)

    return F.mse_loss(ntk_s, ntk_t)

# 5. Train

In [ ]:
def train_net(net,
              teacher,
              device,
              epochs=5,
              batch_size=1,
              lr=0.0001,
              val_percent=0.1,
              save_cp=True,
              img_scale=0.2,
              use_mse=False):
    """
    Train a segmentation network with train/test split, TensorBoard logging,
    LR scheduling, and optional checkpoint saving.

    Args:
        net (torch.nn.Module):
            The segmentation model to train. It is expected to define:
            - net.n_channels: number of input channels
            - net.n_classes : number of output classes (1 for binary segmentation)

        device (torch.device):
            Target device for training (e.g., torch.device("cuda") or torch.device("cpu")).

        epochs (int, optional):
            Number of training epochs. Default is 5.

        batch_size (int, optional):
            Batch size for DataLoader. Default is 1.

        lr (float, optional):
            Learning rate for optimizer. Default is 1e-4.

        val_percent (float, optional):
            Fraction of dataset used for test (0~1). Default is 0.1.

        save_cp (bool, optional):
            Whether to save model checkpoints after each epoch. Default is True.

        img_scale (float, optional):
            Image resizing scale used by dataset loader (e.g., 0.2 means downscale to 20%).
            Default is 0.2.

    Returns:
        None

    Notes:
        - Uses RMSprop optimizer.
        - Uses ReduceLROnPlateau scheduler (monitors test score).
        - Loss:
            * CrossEntropyLoss for multi-class segmentation (n_classes > 1)
            * BCEWithLogitsLoss for binary segmentation (n_classes == 1)
        - Logs training stats and images to TensorBoard.
        - Requires global variables / paths:
            dir_img, dir_mask, dir_checkpoint
        - Requires helper components:
            BasicDataset, eval_net
    """

    dataset = BasicDataset(dir_img, dir_mask, img_scale)

    # Train/val split
    n_val = int(len(dataset) * val_percent)
    n_train = len(dataset) - n_val
    train_set, val_set = random_split(dataset, [n_train, n_val])

    # DataLoader
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=True, drop_last=True)

    # TensorBoard logger
    writer = SummaryWriter(comment=f'LR_{lr}_BS_{batch_size}_SCALE_{img_scale}')
    global_step = 0

    logging.info(f'''Starting training:
        Epochs:          {epochs}
        Batch size:      {batch_size}
        Learning rate:   {lr}
        Training size:   {n_train}
        test size: {n_val}
        Checkpoints:     {save_cp}
        Device:          {device.type}
        Images scaling:  {img_scale}
    ''')

    # Optimizer, scheduler, loss
    optimizer = optim.RMSprop(net.parameters(), lr=lr, weight_decay=1e-8, momentum=0.9)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        'min' if net.n_classes > 1 else 'max',
        patience=2
    )
    # Choose loss function
    if use_mse:
        criterion = nn.MSELoss()
    else:
        criterion = nn.CrossEntropyLoss() if net.n_classes > 1 else nn.BCEWithLogitsLoss()


    # Training loop
    for epoch in range(epochs):
        net.train()
        epoch_loss = 0

        with tqdm(total=n_train, desc=f'Epoch {epoch + 1}/{epochs}', unit='img') as pbar:
            for batch in train_loader:
                imgs = batch['image']
                true_masks = batch['mask']

                # Sanity check for input channels
                assert imgs.shape[1] == net.n_channels, \
                    f'Network has been defined with {net.n_channels} input channels, ' \
                    f'but loaded images have {imgs.shape[1]} channels. Please check that ' \
                    'the images are loaded correctly.'

                # Move data to device
                imgs = imgs.to(device=device, dtype=torch.float32)
                mask_type = torch.float32 if net.n_classes == 1 else torch.long
                true_masks = true_masks.to(device=device, dtype=mask_type)

                # Forward pass
                masks_pred = net(imgs)
                seg_loss = criterion(masks_pred, true_masks)

                # Compute loss
                x_ntk = torch.randn_like(imgs).to(device)

                # NTKD loss（teacher vs net）
                ntkd_loss = compute_ntkd_loss(
                    teacher=teacher,
                    student=net,
                    x_ntk=x_ntk,
                    epsilon=1e-6,
                    R=2,
                    normalize=True
                )

                # loss
                loss = seg_loss + 1e-3 * ntkd_loss
                epoch_loss += loss.item()

                # TensorBoard logging
                writer.add_scalar('Loss/train', loss.item(), global_step)
                pbar.set_postfix({'loss (batch)': loss.item()})

                # Backprop
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_value_(net.parameters(), 0.1)  # gradient clipping
                optimizer.step()

                # Update progress
                pbar.update(imgs.shape[0])
                global_step += 1

                if global_step % (n_train // (3 * batch_size)) == 0:
                    for tag, value in net.named_parameters():
                        tag = tag.replace('.', '/')
                        writer.add_histogram('weights/' + tag, value.data.cpu().numpy(), global_step)
                        writer.add_histogram('grads/' + tag, value.grad.data.cpu().numpy(), global_step)

                    val_score = eval_net(net, val_loader, device)
                    scheduler.step(val_score)

                    writer.add_scalar('learning_rate', optimizer.param_groups[0]['lr'], global_step)

                    if net.n_classes > 1:
                        logging.info(f'cross entropy: {val_score}')
                        writer.add_scalar('Loss/test', val_score, global_step)
                    else:
                        logging.info(f'Dice Coeff: {val_score}')
                        writer.add_scalar('Dice/test', val_score, global_step)

                    # Log sample images/masks
                    writer.add_images('images', imgs, global_step)
                    if net.n_classes == 1:
                        writer.add_images('masks/true', true_masks, global_step)
                        writer.add_images('masks/pred', torch.sigmoid(masks_pred) > 0.5, global_step)

        # Save checkpoint
        if save_cp:
            checkpoint_path = os.path.join(dir_checkpoint, f'CP_epoch{epoch + 1}.pth')
            torch.save(net.state_dict(), checkpoint_path)
            logging.info(f'Checkpoint {epoch + 1} saved!')

    writer.close()


In [ ]:
import os
dir_checkpoint = 'checkpoints/'
dir_img =  'car/imgs/'
dir_mask =  'car/masks/'
# Create the directory if it does not exist
os.makedirs(dir_checkpoint, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Using device {device}')


net = P_UNet()
teacher = UNet()
net.to(device=device)
teacher.to(device=device)

ckpt = torch.load("checkpoints/teacher_model.pt")


teacher.load_state_dict(ckpt["model_state_dict"], strict=True)
teacher.eval()

In [ ]:
train_net(net=net,
              teacher=teacher,
              epochs=120,
              batch_size=8,
              lr=0.0001,
              device=device,
              img_scale=0.2,
              val_percent=10/100)